模板复现类

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Wedge
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.transforms import Bbox

current_dir = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))
CLEAN_DIR = os.path.join(base_dir, "data", "clean")
FIG_DIR = os.path.join(base_dir, "outputs", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

TPL_W, TPL_H, TPL_DPI = 12.0, 9.0, 200
FOOT = "*注：数据来源于 Tableau Superstore 公开样本，统计区间 2015.01–2018.12"

TPL_BG = "#1A1E43"; TPL_TEXT = "#FFFFFF"; TPL_SUB = "#DDDDE3"; TPL_TICK = "#9BA0BC"; TPL_GRID = "#3A4173"
TPL_BLUE = "#0070C0"; TPL_BLUE_D = "#0A4C93"; TPL_BLUE_S = "#1E88C7"; TPL_BLUE2 = "#00B0F0"
TPL_SKY = "#29ABE2"; TPL_CYAN = "#66CBDD"; TPL_TEAL = "#11A7AD"; TPL_GOLD = "#F5C353"
TPL_RED = "#E74E69"; TPL_DARKRED = "#C0504D"; TPL_VIOLET = "#512DCB"; TPL_VIOLET_L = "#6073F3"
TPL_GREY = "#717389"; TPL_GREY_L = "#9EA3BC"; TPL_GREY_D = "#363A59"

F_TITLE, F_SUB, F_TICK, F_LAB, F_FOOT, F_LEG = 40, 30, 22, 22, 17, 21

SUB_CN = {"Phones": "手机", "Chairs": "椅子", "Storage": "收纳", "Tables": "桌子", "Binders": "装订", "Machines": "机器", "Art": "美术", "Appliances": "家电", "Bookcases": "书柜", "Copiers": "复印机", "Furnishings": "家具", "Paper": "纸张", "Accessories": "配件", "Labels": "标签", "Envelopes": "信封", "Fasteners": "紧固件", "Supplies": "耗材"}
SHIP_CN = {"Standard Class": "标准快递", "Second Class": "二级快递", "First Class": "一级快递", "Same Day": "当日达"}
SHORT = {"California": "加州", "New York": "纽约州", "Texas": "德州", "Washington": "华盛顿州", "Pennsylvania": "宾州", "Florida": "佛州", "Illinois": "伊利诺伊", "Ohio": "俄亥俄", "Michigan": "密歇根", "Delaware": "特拉华", "Iowa": "艾奥瓦", "Kansas": "堪萨斯", "Arkansas": "阿肯色", "Missouri": "密苏里", "Nebraska": "内布拉斯加", "Virginia": "弗吉尼亚", "North Carolina": "北卡", "Georgia": "佐治亚", "Arizona": "亚利桑那", "Colorado": "科罗拉多", "Tennessee": "田纳西", "Oregon": "俄勒冈", "Wisconsin": "威斯康星", "Minnesota": "明尼苏达", "Massachusetts": "马萨诸塞", "Indiana": "印第安纳"}

def _clean(ax, grid_y=False):
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(length=0, labelsize=F_TICK, colors=TPL_TICK, pad=12)
    ax.xaxis.grid(False)
    if grid_y:
        ax.set_axisbelow(True)
        ax.yaxis.grid(True, color=TPL_GRID, lw=1.0, ls=(0, (7, 7)))

def fit(text, base=F_TITLE, width=0.86):
    fs = base
    while len(text) * fs / 72.0 > TPL_W * width and fs > 16: fs -= 1
    return fs

def canvas(title, sub=None, box=(0.105, 0.155, 0.855, 0.53), center=False, foot=True):
    fig = plt.figure(figsize=(TPL_W, TPL_H), dpi=TPL_DPI)
    fig.patch.set_facecolor(TPL_BG)
    x, ha = (0.5, "center") if center else (0.055, "left")
    fig.text(x, 0.935, title, ha=ha, va="top", fontsize=fit(title), fontweight="bold", color=TPL_TEXT)
    if sub: fig.text(x, 0.850, sub, ha=ha, va="top", fontsize=fit(sub, F_SUB, 0.94), color=TPL_SUB)
    if foot: fig.text(0.055, 0.045, FOOT, ha="left", va="bottom", fontsize=F_FOOT, color=TPL_TICK)
    ax = fig.add_axes(box)
    ax.set_facecolor(TPL_BG)
    _clean(ax)
    return fig, ax

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=TPL_DPI, facecolor=TPL_BG, bbox_inches=Bbox.from_bounds(0, 0, TPL_W, TPL_H))
    plt.close(fig)

def vgrad(ax, x, heights, width, top, bottom, zorder=3):
    cmap = LinearSegmentedColormap.from_list("vg", [bottom, top])
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    hs = np.asarray(heights, dtype=float)
    bars = ax.bar(x, hs, width=width, color=bottom, edgecolor="none", zorder=zorder)
    g = np.linspace(0, 1, 256).reshape(-1, 1)
    for b in bars:
        h = b.get_height()
        im = ax.imshow(g, aspect="auto", cmap=cmap, origin="lower", zorder=zorder + 0.1, extent=[b.get_x(), b.get_x() + b.get_width(), min(0.0, h), max(0.0, h)])
        im.set_clip_path(b)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    return bars

def hgrad(ax, x0, width, yc, height, left, right, zorder=3):
    cmap = LinearSegmentedColormap.from_list("hg", [left, right])
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    r = Rectangle((x0, yc - height / 2), width, height, facecolor=left, edgecolor="none", zorder=zorder)
    ax.add_patch(r)
    g = np.linspace(0, 1, 256).reshape(1, -1)
    im = ax.imshow(g, aspect="auto", cmap=cmap, origin="lower", zorder=zorder + 0.1, extent=[x0, x0 + width, yc - height / 2, yc + height / 2])
    im.set_clip_path(r)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    return r

def cap(ax, x0, x1, y, height, color, zorder=3):
    y0, y1 = ax.get_ylim()
    h_in = ax.get_position().height * TPL_H
    lw = max(2.0, height / abs(y1 - y0) * h_in * 72)
    ax.plot([x0, x1], [y, y], color=color, lw=lw, solid_capstyle="round", zorder=zorder)

def tw(label, fs):
    return sum((1.00 if ord(c) > 127 else 0.55) for c in label) * fs / 864.0

def leader(mid_deg, rx, ry):
    c, sn = np.cos(np.deg2rad(mid_deg)), np.sin(np.deg2rad(mid_deg))
    yy = ry * np.sign(sn) * max(0.50, abs(sn))
    return (rx if c >= 0 else -rx), yy

def legend_row(fig, x, y, items, fs=F_LEG, mark="square"):
    xx = x
    for label, color in items:
        if mark == "square":
            fig.add_artist(Rectangle((xx, y - 0.008), 0.016, 0.016, transform=fig.transFigure, facecolor=color, edgecolor="none"))
        else:
            fig.add_artist(Circle((xx + 0.008, y), 0.010, transform=fig.transFigure, facecolor=color, edgecolor=TPL_BG, lw=1.0))
        fig.text(xx + 0.024, y, label, ha="left", va="center", fontsize=fs, color=TPL_SUB)
        xx += 0.024 + tw(label, fs) + 0.030

df = pd.read_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), parse_dates=["Order Date"])

def fig01_gradient_bar(df):
    g = df.groupby("State")["Sales"].sum().nlargest(6)
    fig, ax = canvas("各州销售额分布", "加州销售额最高，占全国总额的" + f"{g.iloc[0] / df['Sales'].sum() * 100:.1f}%", box=(0.115, 0.155, 0.845, 0.515))
    x = np.arange(len(g)); ax.set_xlim(-0.62, len(g) - 0.38); ax.set_ylim(0, g.max() * 1.28)
    yt = np.arange(0, g.max() * 1.02, 100000); ax.set_yticks(yt); ax.set_yticklabels([f"{v / 1e4:.0f}" for v in yt])
    _clean(ax, grid_y=True); ax.set_xticks(x); ax.set_xticklabels([SHORT.get(n, n) for n in g.index], fontsize=F_TICK)
    vgrad(ax, x, g.values, 0.30, TPL_SKY, TPL_BLUE_D)
    for xi, v in zip(x, g.values): ax.text(xi, v * 1.018, f"{v / 1e4:.0f}", ha="center", va="bottom", fontsize=F_LAB, color=TPL_TEXT)
    save(fig, "fig01_各州销售额渐变柱形图")

def fig02_butterfly(df):
    p = df[df["Order Year"].isin([2017, 2018])].groupby(["State", "Order Year"])["Sales"].sum().unstack("Order Year").dropna().nlargest(5, 2018)
    vmax = float(p.values.max()); gap = 0.13
    fig, ax = canvas("各州两年销售额对比", "两端为两年销售额（万元）", box=(0.02, 0.125, 0.96, 0.575))
    ax.set_xlim(-1.02, 1.02); ax.set_ylim(len(p) - 0.45, -0.75)
    for i, (name, r) in enumerate(p.iterrows()):
        w17 = (1 - gap) * r[2017] / vmax; w18 = (1 - gap) * r[2018] / vmax
        ax.barh(i, -w18, left=-gap, height=0.52, color=TPL_BLUE_S, edgecolor="none")
        ax.barh(i, w17, left=gap, height=0.52, color=TPL_RED, edgecolor="none")
        ax.text(-gap - w18 + 0.035, i, f"{r[2018] / 1e4:.0f}", ha="left", va="center", fontsize=F_LAB, color=TPL_TEXT)
        ax.text(gap + w17 - 0.035, i, f"{r[2017] / 1e4:.0f}", ha="right", va="center", fontsize=F_LAB, color=TPL_TEXT)
        ax.text(0, i, SHORT.get(name, name), ha="center", va="center", fontsize=F_TICK + 1, color=TPL_TEXT)
    ax.set_xticks([]); ax.set_yticks([])
    fig.text(0.5 - 0.16, 0.762, "2018", ha="center", va="bottom", fontsize=F_LEG, color=TPL_BLUE_S, fontweight="bold")
    fig.text(0.5 + 0.16, 0.762, "2017", ha="center", va="bottom", fontsize=F_LEG, color=TPL_RED, fontweight="bold")
    save(fig, "fig02_各州两年销售额对比蝴蝶图")

def fig03_gantt(df):
    win = df[df["Order Date"] >= "2017-07-01"]
    g = win.groupby(["Customer ID", "Customer Name"])
    c = pd.DataFrame({"first": g["Order Date"].min(), "last": g["Order Date"].max(), "orders": g["Order ID"].nunique()}).reset_index().nlargest(7, "orders").sort_values("first")
    t0 = c["first"].min(); c["s"] = (c["first"] - t0).dt.days.astype(float); c["dur"] = (c["last"] - c["first"]).dt.days.astype(float)
    c["dur"] = c["dur"].clip(lower=30); months = np.maximum(c["dur"] / 30.0, 1.0); c["pct"] = np.clip(c["orders"] / months, 0.12, 1.0)
    span = float((c["s"] + c["dur"]).max())
    fig, ax = canvas("TOP7 客户订单周期进度", "条长为该客户首单至末单跨度，条内为期间活跃月份占比", box=(0.20, 0.105, 0.755, 0.585), center=True)
    ax.set_xlim(-span * 0.05, span * 1.06); ax.set_ylim(len(c) - 0.4, -1.0)
    for i, (_, r) in enumerate(c.iterrows()):
        w = r["dur"] * r["pct"]
        hgrad(ax, r["s"], w, i, 0.50, TPL_SKY, TPL_BLUE_D, zorder=3)
        ax.text(r["s"] + w / 2, i, f"{r['pct'] * 100:.0f}%", ha="center", va="center", fontsize=F_LAB - 2, color=TPL_TEXT, zorder=5)
        ax.text(-span * 0.03, i, str(r["Customer Name"])[:13], ha="right", va="center", fontsize=F_TICK - 3, color=TPL_TEXT, clip_on=False)
    n = 4; ticks = [span * 0.96 * k / (n - 1) for k in range(n)]
    for t in ticks: ax.plot([t, t], [-0.95, len(c) - 0.55], color=TPL_GREY, lw=0.9, ls=(0, (6, 6)), zorder=1)
    ax.set_xticks(ticks); ax.set_xticklabels([(t0 + pd.Timedelta(days=d)).strftime("%Y/%m/%d") for d in ticks], fontsize=F_TICK - 3)
    ax.xaxis.set_ticks_position("top"); ax.tick_params(axis="x", pad=16); ax.set_yticks([])
    save(fig, "fig03_TOP7客户订单周期进度甘特图")

def fig04_smooth_line(df):
    from scipy.interpolate import PchipInterpolator
    w = df[(df["Order Date"] >= "2017-05-01") & (df["Order Date"] < "2018-06-01")]
    m = w.groupby("Order YM")["Sales"].sum().sort_index(); x = np.arange(len(m)); xs = np.linspace(0, len(m) - 1, 700)
    ys = PchipInterpolator(x, m.values)(xs); vmax = float(m.max())
    fig, ax = canvas("月度销售额平滑走势", f"2017 年 5 月至 2018 年 5 月，{int(str(m.idxmax())[-2:])} 月达到峰值 {vmax / 1e4:.0f} 万", box=(0.115, 0.245, 0.825, 0.50))
    ax.set_ylim(-vmax * 0.36, vmax * 1.18); ax.set_xlim(-0.6, len(m) - 0.4); ax.plot(xs, ys, color=TPL_DARKRED, lw=2.2, zorder=4, solid_capstyle="round")
    i = int(np.argmax(m.values)); ax.plot([i, i], [0, m.values[i] * 0.97], color=TPL_DARKRED, lw=1.3, ls=(0, (5, 5)), zorder=2)
    ax.text(i, m.values[i] * 1.03, f"{m.values[i] / 1e4:.0f}", ha="center", va="bottom", fontsize=F_LAB, color=TPL_TEXT)
    yt = np.arange(0, vmax * 1.02, 20000); ax.set_yticks(yt); ax.set_yticklabels([f"{v / 1e4:.0f}" for v in yt]); ax.tick_params(axis="y", colors=TPL_DARKRED)
    ax.set_xticks(x); ax.set_xticklabels([f"{int(str(v)[-2:])}月" for v in m.index], fontsize=F_TICK)
    ax.spines["bottom"].set_visible(True); ax.spines["bottom"].set_position(("data", 0)); ax.spines["bottom"].set_color(TPL_GREY_L); ax.spines["bottom"].set_linewidth(1.2); ax.tick_params(axis="x", colors=TPL_SUB, pad=8)
    yb = -vmax * 0.255; hh = vmax * 0.092
    cap(ax, -0.4, 7.6, yb, hh, TPL_CYAN, zorder=6); cap(ax, 7.6, len(m) - 0.4, yb, hh, TPL_GOLD, zorder=6)
    ax.text(3.6, yb, "2017", ha="center", va="center", fontsize=F_LAB - 8, color=TPL_BG, zorder=7, fontweight="bold")
    ax.text(9.8, yb, "2018", ha="center", va="center", fontsize=F_LAB - 8, color=TPL_BG, zorder=7, fontweight="bold")
    save(fig, "fig04_月度销售额平滑折线图")

def fig05_compare_line(df):
    p = df[df["Order Year"].isin([2017, 2018]) & (df["Order Month"] <= 6)].groupby(["Order Month", "Order Year"])["Sales"].sum().unstack("Order Year").dropna()
    fig, ax = canvas("上半年各月销售额同比", "2018 年 5 月同比增幅最大，单月同比增长 37.3%", box=(0.115, 0.155, 0.845, 0.505))
    ax.set_xlim(0.4, len(p) + 0.6); ax.set_ylim(0, p.values.max() * 1.35)
    yt = np.arange(0, p.values.max() * 1.12, 20000); ax.set_yticks(yt); ax.set_yticklabels([f"{v / 1e4:.0f}" for v in yt]); _clean(ax, grid_y=True)
    ax.plot(p.index, p[2017], color=TPL_RED, lw=2.4, marker="o", ms=8, mfc=TPL_RED, mec=TPL_RED, zorder=4)
    ax.plot(p.index, p[2018], color=TPL_BLUE_S, lw=2.4, marker="o", ms=8, mfc=TPL_BLUE_S, mec=TPL_BLUE_S, zorder=5)
    for m_, v in p[2017].items(): ax.text(m_, v * 0.905, f"{v / 1e4:.0f}", ha="center", va="top", fontsize=F_LAB - 3, color=TPL_SUB)
    for m_, v in p[2018].items(): ax.text(m_, v * 1.075, f"{v / 1e4:.0f}", ha="center", va="bottom", fontsize=F_LAB - 3, color=TPL_SUB)
    ax.set_xticks(list(p.index)); ax.set_xticklabels([f"{int(m)}月" for m in p.index], fontsize=F_TICK, color=TPL_SUB)
    legend_row(fig, 0.60, 0.782, [("2017 年", TPL_RED), ("2018 年", TPL_BLUE_S)], mark="dot")
    save(fig, "fig05_上半年各月销售额同比对比折线图")

def fig06_water_ball(df):
    cust = df.groupby("Customer ID")["Sales"].sum().sort_values(ascending=False)
    rate = float(np.clip(cust.head(max(1, int(len(cust) * 0.2))).sum() / cust.sum(), 0.05, 0.95))
    fig = plt.figure(figsize=(TPL_W, TPL_H), dpi=TPL_DPI); fig.patch.set_facecolor(TPL_BG)
    fig.text(0.055, 0.935, "TOP20% 客户销售集中度", ha="left", va="top", fontsize=fit("TOP20% 客户销售集中度"), fontweight="bold", color=TPL_TEXT)
    fig.text(0.055, 0.850, f"{int(len(cust) * 0.2)} 位头部客户贡献了全部销售额的 {rate * 100:.0f}%", ha="left", va="top", fontsize=F_SUB - 3, color=TPL_SUB)
    fig.text(0.055, 0.045, FOOT, ha="left", va="bottom", fontsize=F_FOOT, color=TPL_TICK)
    ax = fig.add_axes([0.22, 0.085, 0.56, 0.675]); ax.set_xlim(-1.16, 1.16); ax.set_ylim(-1.16, 1.16); ax.set_aspect("equal"); ax.axis("off"); ax.set_facecolor(TPL_BG)
    ax.add_patch(Circle((0, 0), 1.06, facecolor="none", edgecolor=TPL_BLUE, lw=2.2, zorder=5))
    ax.add_patch(Circle((0, 0), 1.00, facecolor="none", edgecolor=TPL_BLUE, lw=4.0, zorder=5))
    ball = Circle((0, 0), 1.00, facecolor=TPL_BG, edgecolor="none", zorder=2); ax.add_patch(ball)
    water = Rectangle((-1.0, -1.0), 2.0, 2.0 * rate, facecolor=TPL_BLUE, edgecolor="none", zorder=3); ax.add_patch(water); water.set_clip_path(ball)
    ax.text(0, -1.0 + 2.0 * rate * 0.50, f"{rate * 100:.0f}%", ha="center", va="center", fontsize=64, color=TPL_TEXT, zorder=6)
    save(fig, "fig06_TOP20客户销售集中度水球图")

def fig07_jade_ring(df):
    cust = df.groupby("Customer ID")["Sales"].sum().sort_values(ascending=False)
    tot, n = cust.sum(), len(cust); cuts = [0.0, 0.10, 0.30, 0.60, 1.0]; names = ["TOP10% 客户", "10–30% 客户", "30–60% 客户", "60–100% 客户"]
    vals = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        lo, hi = int(n * a), max(int(n * b), int(n * a) + 1); vals.append(cust.iloc[lo:hi].sum() / tot * 100)
    colors = [TPL_RED, TPL_GOLD, TPL_TEAL, TPL_SKY]
    fig = plt.figure(figsize=(TPL_W, TPL_H), dpi=TPL_DPI); fig.patch.set_facecolor(TPL_BG)
    fig.text(0.055, 0.935, "客户销售集中度（玉玦图）", ha="left", va="top", fontsize=fit("客户销售集中度（玉玦图）"), fontweight="bold", color=TPL_TEXT)
    fig.text(0.055, 0.850, "弧长即该档客户的销售占比，四层由外向内依次递减", ha="left", va="top", fontsize=fit("弧长即该档客户的销售占比，四层由外向内依次递减", F_SUB, 0.94), color=TPL_SUB)
    fig.text(0.055, 0.045, FOOT, ha="left", va="bottom", fontsize=F_FOOT, color=TPL_TICK)
    ax = fig.add_axes([0.205, -0.075, 0.60, 0.80]); ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35); ax.set_aspect("equal"); ax.axis("off"); ax.set_facecolor(TPL_BG)
    for i, (v, c) in enumerate(zip(vals, colors)):
        r_out = 1.15 - i * 0.245; arc = v * 3.6
        ax.add_patch(Wedge((0, 0), r_out, 0, arc, width=0.150, facecolor=c, edgecolor="none", zorder=3))
        th = np.deg2rad(arc * 0.55); rr = r_out - 0.075
        ax.text(rr * np.cos(th), rr * np.sin(th), f"{v:.1f}%", ha="center", va="center", fontsize=F_LAB - 2, color=TPL_TEXT, zorder=5)
    for i, nm in enumerate(names): fig.text(0.078, 0.552 - i * 0.048, nm, ha="left", va="center", fontsize=F_LEG - 3, color=colors[i])
    save(fig, "fig07_客户销售集中度玉玦图")

def fig08_nightingale(df):
    g = df.groupby("Sub-Category")["Sales"].sum().nlargest(6)
    colors = [TPL_RED, TPL_GOLD, TPL_TEAL, TPL_SKY, TPL_BLUE_S, TPL_VIOLET]
    tot = g.sum(); radii = 0.32 + 0.74 * (g.values - g.values.min()) / (g.values.max() - g.values.min())
    fig = plt.figure(figsize=(TPL_W, TPL_H), dpi=TPL_DPI); fig.patch.set_facecolor(TPL_BG)
    fig.text(0.055, 0.935, "六子类目销售规模（南丁格尔玫瑰）", ha="left", va="top", fontsize=fit("六子类目销售规模（南丁格尔玫瑰）"), fontweight="bold", color=TPL_TEXT)
    fig.text(0.055, 0.850, "扇形角度按销售额分配，半径按销售额线性缩放", ha="left", va="top", fontsize=F_SUB, color=TPL_SUB)
    fig.text(0.055, 0.045, FOOT, ha="left", va="bottom", fontsize=F_FOOT, color=TPL_TICK)
    ax = fig.add_axes([0.20, 0.03, 0.60, 0.82]); ax.set_xlim(-1.78, 1.78); ax.set_ylim(-1.50, 1.50); ax.set_aspect("equal"); ax.axis("off"); ax.set_facecolor(TPL_BG)
    a = 90.0
    for i, (name, v, r, c) in enumerate(zip(g.index, g.values, radii, colors)):
        w = v / tot * 360.0
        ax.add_patch(Wedge((0, 0), r, a - w + 1.0, a - 1.0, facecolor=c, edgecolor=TPL_BG, lw=2.0, zorder=3))
        mid = np.deg2rad(a - w / 2); ex, ey = r * np.cos(mid), r * np.sin(mid); tx, ty = leader(a - w / 2, 1.28, 1.34)
        ax.annotate(f"{SUB_CN.get(name, name)}\n{v / 1e4:.0f}万", xy=(ex, ey), xytext=(tx, ty), ha="left" if ex >= 0 else "right", va="center", fontsize=F_LAB - 1, color=c, zorder=6, arrowprops=dict(arrowstyle="-", color=c, lw=1.4, shrinkA=0, shrinkB=4, connectionstyle="angle,angleA=0,angleB=90,rad=0"))
        a -= w
    save(fig, "fig08_六子类目销售规模南丁格尔玫瑰图")

def fig09_nightingale_donut(df):
    g = df.groupby("Ship Mode")["Sales"].sum().sort_values(ascending=False)
    colors = [TPL_VIOLET, TPL_VIOLET_L, TPL_CYAN, TPL_GOLD]; tot = g.sum(); shares = g / tot * 100
    fig = plt.figure(figsize=(TPL_W, TPL_H), dpi=TPL_DPI); fig.patch.set_facecolor(TPL_BG)
    fig.text(0.055, 0.935, "各配送方式销售贡献", ha="left", va="top", fontsize=fit("各配送方式销售贡献"), fontweight="bold", color=TPL_TEXT)
    fig.text(0.055, 0.850, "环宽固定，弧长按销售额分配，标签以折线引出", ha="left", va="top", fontsize=F_SUB, color=TPL_SUB)
    fig.text(0.055, 0.045, FOOT, ha="left", va="bottom", fontsize=F_FOOT, color=TPL_TICK)
    ax = fig.add_axes([0.22, 0.02, 0.58, 0.84]); ax.set_xlim(-1.95, 1.95); ax.set_ylim(-1.55, 1.55); ax.set_aspect("equal"); ax.axis("off"); ax.set_facecolor(TPL_BG)
    a = 90.0
    for i, (name, v, c) in enumerate(zip(shares.index, shares.values, colors)):
        w = v * 3.6
        ax.add_patch(Wedge((0, 0), 1.00, a - w + 0.8, a - 0.8, width=0.42, facecolor=c, edgecolor="none", zorder=3))
        for rr in (0.74, 0.85, 0.95): ax.add_patch(Wedge((0, 0), rr, a - w + 0.8, a - 0.8, width=0.001, facecolor="none", edgecolor=TPL_BG, lw=1.0, zorder=4))
        mid = np.deg2rad(a - w / 2); ex, ey = 1.00 * np.cos(mid), 1.00 * np.sin(mid); tx, ty = leader(a - w / 2, 1.30, 1.20)
        ax.annotate(f"{SHIP_CN.get(name, name)}\n{v:.1f}%", xy=(ex, ey), xytext=(tx, ty), ha="left" if ex >= 0 else "right", va="center", fontsize=F_LAB - 1, color=c, zorder=6, arrowprops=dict(arrowstyle="-", color=c, lw=1.4, shrinkA=0, shrinkB=3, connectionstyle="angle,angleA=0,angleB=90,rad=0"))
        a -= w
    save(fig, "fig09_各配送方式销售贡献南丁格尔圆环图")

def fig10_gauge(df):
    v = df["Profit"].sum() / df["Sales"].sum() * 100; vlo, vhi = 0.0, 30.0
    def ang(val): return 210.0 - (val - vlo) / (vhi - vlo) * 240.0
    fig = plt.figure(figsize=(TPL_W, TPL_H), dpi=TPL_DPI); fig.patch.set_facecolor(TPL_BG)
    fig.text(0.5, 0.912, "综合净利率健康度仪表盘", ha="center", va="top", fontsize=fit("综合净利率健康度仪表盘"), fontweight="bold", color=TPL_TEXT)
    ax = fig.add_axes([0.235, 0.02, 0.53, 0.80]); ax.set_xlim(-1.28, 1.28); ax.set_ylim(-1.30, 1.30); ax.set_aspect("equal"); ax.axis("off"); ax.set_facecolor(TPL_BG)
    ax.add_patch(Wedge((0, 0), 1.12, ang(vhi), ang(vlo), width=0.155, facecolor=TPL_BLUE, edgecolor="none", zorder=2))
    for lo, hi, c in [(0, 9, "#087E8B"), (9, 18, "#FFC54D"), (18, 30, "#F94C66")]: ax.add_patch(Wedge((0, 0), 0.90, ang(hi), ang(lo), width=0.135, facecolor=c, edgecolor="none", zorder=3))
    for tv in np.arange(vlo, vhi + 0.01, 3.0):
        th = np.deg2rad(ang(tv)); ax.plot([1.00 * np.cos(th), 1.125 * np.cos(th)], [1.00 * np.sin(th), 1.125 * np.sin(th)], color=TPL_BG, lw=2.0, zorder=5)
        ax.text(0.700 * np.cos(th), 0.700 * np.sin(th), f"{tv:.0f}", ha="center", va="center", fontsize=F_LAB - 3, color=TPL_SUB, zorder=5)
    th = np.deg2rad(ang(v)); ax.plot([0.06 * np.cos(th), 0.90 * np.cos(th)], [0.06 * np.sin(th), 0.90 * np.sin(th)], color=TPL_TEXT, lw=2.4, solid_capstyle="round", zorder=7)
    ax.text(0.02, -0.40, f"{v:.1f}", ha="center", va="center", fontsize=52, color=TPL_TEXT, zorder=7)
    save(fig, "fig10_综合净利率健康度仪表盘")

def fig11_bar_line(df):
    y = df.groupby("Order Year")["Sales"].sum().to_frame()
    y["YoY"] = y["Sales"].pct_change() * 100
    fig, ax = canvas("近四年销售额及增长率", "销售额由 48 万增至 73 万，2017 年增速最快（+29.5%）", box=(0.105, 0.155, 0.855, 0.505))
    x = np.arange(len(y)); ax.set_ylim(0, y["Sales"].max() * 2.40); ax.set_xlim(-0.62, len(y) - 0.38); _clean(ax)
    vgrad(ax, x, y["Sales"].values, 0.30, TPL_SKY, TPL_BLUE_D)
    for xi, v in zip(x, y["Sales"]): ax.text(xi, v * 1.018, f"{v / 1e4:.0f}", ha="center", va="bottom", fontsize=F_LAB, color=TPL_TEXT, zorder=4)
    ax.set_yticks([]); ax.set_xticks(x); ax.set_xticklabels(y.index.astype(int).astype(str), fontsize=F_TICK, color=TPL_SUB)
    ax2 = ax.twinx()
    for sp in ax2.spines.values(): sp.set_visible(False)
    ax2.set_yticks([]); ax2.tick_params(length=0); ax2.set_ylim(-67, 61)
    ax2.plot(x, y["YoY"].fillna(0).values, color=TPL_RED, lw=2.4, marker="o", ms=9, mfc=TPL_TEXT, mec=TPL_RED, mew=2.4, zorder=6)
    for xi, vv in zip(x, y["YoY"]):
        if pd.notna(vv): ax2.text(xi, vv + 7.0, f"{vv:.0f}%", ha="center", va="bottom", fontsize=F_LAB, color=TPL_SUB)
    legend_row(fig, 0.560, 0.762, [("销售额", TPL_BLUE_S), ("同比", TPL_RED)], mark="dot")
    save(fig, "fig11_近四年销售额及增长率柱形折线图")

def fig12_target_bar(df):
    p = df[df["Order Year"].isin([2017, 2018])].groupby(["Sub-Category", "Order Year"])["Sales"].sum().unstack("Order Year").dropna()
    act = p[2018].nlargest(6); tgt = (p[2017] * 1.15).reindex(act.index)
    fig, ax = canvas("各子类目销量 vs 目标", "防晒整体销量最好，目标框为上年上浮 15%", box=(0.105, 0.155, 0.855, 0.505))
    x = np.arange(len(act)); vmax = max(act.max(), tgt.max()); ax.set_ylim(0, vmax * 1.22); ax.set_xlim(-0.62, len(act) - 0.38); _clean(ax)
    ax.bar(x, tgt.values, width=0.44, color="none", edgecolor=TPL_GREY_L, lw=2.0, zorder=2)
    ax.bar(x, act.values, width=0.20, color=TPL_BLUE, edgecolor="none", zorder=3)
    for xi, v in zip(x, act.values): ax.text(xi, max(v, tgt.iloc[xi]) * 1.035, f"{v / 1e4:.0f}", ha="center", va="bottom", fontsize=F_LAB, color=TPL_TEXT, zorder=5)
    ax.set_yticks([]); ax.set_xticks(x); ax.set_xticklabels([SUB_CN.get(n, n) for n in act.index], fontsize=F_TICK, color=TPL_SUB)
    legend_row(fig, 0.105, 0.762, [("目标销量", TPL_GREY_L), ("实际销量", TPL_BLUE)])
    save(fig, "fig12_各子类目销量vs目标柱形图")

def fig13_bullet(df):
    piv = df.pivot_table(index="Sub-Category", columns="Segment", values="Sales", aggfunc="sum")
    top = piv.sum(axis=1).nlargest(6); seg = piv.loc[top.index]; tgt = top * 0.92
    fig, ax = canvas("各子类目分客群销量构成", "三段为个人 / 企业 / 家庭三类客户，金线为年度目标", box=(0.105, 0.155, 0.855, 0.505))
    x = np.arange(len(top)); ax.set_ylim(0, top.max() * 1.18); ax.set_xlim(-0.62, len(top) - 0.38); _clean(ax)
    ax.bar(x, top.values * 1.10, width=0.32, color="#2C3260", edgecolor="none", zorder=1)
    ax.bar(x, seg["Consumer"].values, width=0.32, color=TPL_BLUE_S, edgecolor="none", zorder=2)
    ax.bar(x, seg["Corporate"].values, width=0.32, color=TPL_CYAN, edgecolor="none", bottom=seg["Consumer"].values, zorder=3)
    ax.bar(x, seg["Home Office"].values, width=0.32, color=TPL_BLUE_D, edgecolor="none", bottom=(seg["Consumer"] + seg["Corporate"]).values, zorder=4)
    for xi, v in zip(x, tgt.values): ax.plot([xi - 0.18, xi + 0.18], [v, v], color=TPL_GOLD, lw=3.0, solid_capstyle="butt", zorder=6)
    ax.set_yticks([]); ax.set_xticks(x); ax.set_xticklabels([SUB_CN.get(n, n) for n in top.index], fontsize=F_TICK, color=TPL_SUB)
    legend_row(fig, 0.105, 0.762, [("个人", TPL_BLUE_S), ("企业", TPL_CYAN), ("家庭", TPL_BLUE_D), ("目标", TPL_GOLD)])
    save(fig, "fig13_各子类目分客群销量构成子弹图")

def fig14_slide_bead(df):
    allg = (df.groupby("Sub-Category")["Profit"].sum() / df.groupby("Sub-Category")["Sales"].sum()).sort_values(ascending=False)
    g = allg.head(9).sort_values()
    fig, ax = canvas("各子类目利润率（滑珠图）", "轨道长度固定，滑珠位置即该子类目利润率", box=(0.235, 0.135, 0.665, 0.535))
    vmax = float(g.max()) * 1.04; ax.set_xlim(-vmax * 0.02, vmax); ax.set_ylim(-0.65, len(g) - 0.35); hh = 0.46
    for i, (name, v) in enumerate(g.items()):
        cap(ax, 0.0, vmax, i, hh, TPL_GREY, zorder=2); cap(ax, 0.0, v, i, hh, TPL_BLUE, zorder=3)
        ax.scatter(v, i, s=430, color=TPL_BG, edgecolor="none", lw=0, zorder=5)
        ax.scatter(v, i, s=340, color=TPL_TEXT, edgecolor=TPL_BLUE, lw=4.0, zorder=6)
        ax.text(v - vmax * 0.070, i, f"{v * 100:.0f}%", ha="right", va="center", fontsize=F_LAB, color=TPL_TEXT, zorder=7)
    ax.set_yticks(range(len(g))); ax.set_yticklabels([SUB_CN.get(n, n) for n in g.index], fontsize=F_TICK - 2, color=TPL_SUB); ax.set_xticks([]); ax.tick_params(axis="y", pad=14)
    save(fig, "fig14_各子类目利润率滑珠图")

def fig15_compare_bead(df):
    d = df[df["Order Year"].isin([2017, 2018])]
    p = (d.groupby(["State", "Order Year"])["Profit"].sum() / d.groupby(["State", "Order Year"])["Sales"].sum()).unstack("Order Year")
    p = p.dropna(); p = p[~p.index.isin(["District of Columbia"])].nlargest(5, 2018)
    fig, ax = canvas("各州利润率同比", "蓝珠为 2018 年，灰珠为 2017 年", box=(0.175, 0.155, 0.755, 0.505))
    n = len(p); xmax = float(p.values.max()) * 1.10; ax.set_xlim(0, xmax); ax.set_ylim(n - 0.4, -0.85)
    for i, (name, r) in enumerate(p.iterrows()):
        ax.barh(i, xmax, height=0.26, color=TPL_GREY_D, edgecolor="none", zorder=1)
        ax.barh(i, max(r[2017], r[2018]), height=0.26, color=TPL_BLUE, edgecolor="none", zorder=2)
        ax.scatter(r[2017], i, s=340, color=TPL_GREY_L, edgecolor=TPL_BG, lw=3.0, zorder=5)
        ax.scatter(r[2018], i, s=340, color=TPL_TEXT, edgecolor=TPL_BLUE, lw=4.0, zorder=6)
        ax.text(r[2018], i - 0.30, f"{r[2018] * 100:.0f}%", ha="center", va="bottom", fontsize=F_LAB - 3, color=TPL_SUB, zorder=7)
    ax.set_yticks(range(n)); ax.set_yticklabels([SHORT.get(n, n) for n in p.index], fontsize=F_TICK - 3, color=TPL_SUB); ax.tick_params(axis="y", pad=12)
    ax.set_xticks(np.arange(0, xmax, xmax / 5)); ax.set_xticklabels([f"{v * 100:.0f}%" for v in np.arange(0, xmax, xmax / 5)], fontsize=F_TICK - 2, color=TPL_SUB)
    legend_row(fig, 0.085, 0.762, [("2018 年", TPL_BLUE), ("2017 年", TPL_GREY_L)], mark="dot")
    save(fig, "fig15_各州利润率同比对比滑珠图")

# ================= 4. 执行 =================
if __name__ == "__main__":
    funcs = [fig01_gradient_bar, fig02_butterfly, fig03_gantt, fig04_smooth_line, fig05_compare_line, 
             fig06_water_ball, fig07_jade_ring, fig08_nightingale, fig09_nightingale_donut, 
             fig10_gauge, fig11_bar_line, fig12_target_bar, fig13_bullet, fig14_slide_bead, fig15_compare_bead]
    for fn in funcs:
        try: fn(df)
        except Exception as e: print(f"  !! {fn.__name__} 失败：{type(e).__name__}: {e}")
    print("模板图完成")

模板图完成


业务模型图

In [5]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle, Circle
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.transforms import Bbox

current_di是r = os.getcwd()
base_dir = os.path.abspath(os.path.join(current_dir, ".."))
CLEAN_DIR = os.path.join(base_dir, "data", "clean")
FIG_DIR = os.path.join(base_dir, "outputs", "figures")
TAB_DIR = os.path.join(base_dir, "outputs", "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

TPL_W, TPL_H, TPL_DPI = 12.0, 9.0, 200

FOOT = "*注：数据来源于 Tableau Superstore 公开样本，统计区间 2015.01–2018.12"

TPL_BG = "#1A1E43"; TPL_TEXT = "#FFFFFF"; TPL_SUB = "#DDDDE3"; TPL_TICK = "#9BA0BC"; TPL_GRID = "#3A4173"
TPL_BLUE = "#0070C0"; TPL_BLUE_D = "#0A4C93"; TPL_BLUE_S = "#1E88C7"; TPL_BLUE_L = "#82ADD7"
TPL_SKY = "#29ABE2"; TPL_CYAN = "#66CBDD"; TPL_TEAL = "#11A7AD"; TPL_GOLD = "#F5C353"; TPL_RED = "#E74E69"; TPL_GREY = "#717389"

F_TITLE, F_SUB, F_TICK, F_LAB, F_FOOT, F_LEG = 40, 30, 22, 22, 17, 21
SUB_CN = {"Phones": "手机", "Chairs": "椅子", "Storage": "收纳", "Tables": "桌子", "Binders": "装订", "Machines": "机器", "Art": "美术", "Appliances": "家电", "Bookcases": "书柜", "Copiers": "复印机", "Furnishings": "家具", "Paper": "纸张", "Accessories": "配件", "Labels": "标签", "Envelopes": "信封", "Fasteners": "紧固件", "Supplies": "耗材"}
SUB_SHORT = dict(SUB_CN, **{"Copiers": "复印", "Fasteners": "紧固"})
RFM_SHORT = {"重要价值客户": "价值", "重要唤回客户": "唤回", "重要发展客户": "发展", "重要挽留客户": "挽留", "一般价值客户": "价值", "一般保持客户": "保持", "一般发展客户": "发展", "一般挽留客户": "挽留"}
RFM_COLORS = {"重要价值客户": TPL_BLUE, "重要唤回客户": TPL_SKY, "重要发展客户": TPL_TEAL, "重要挽留客户": TPL_GOLD, "一般价值客户": TPL_BLUE_L, "一般保持客户": TPL_CYAN, "一般发展客户": TPL_GREY, "一般挽留客户": TPL_RED}

def _clean(ax, grid_y=False):
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.tick_params(length=0, labelsize=F_TICK, colors=TPL_TICK, pad=12)
    ax.xaxis.grid(False)
    if grid_y:
        ax.set_axisbelow(True); ax.yaxis.grid(True, color=TPL_GRID, lw=1.0, ls=(0, (7, 7)))

def fit(text, base=F_TITLE, width=0.86):
    fs = base
    while len(text) * fs / 72.0 > TPL_W * width and fs > 16: fs -= 1
    return fs

def canvas(title, sub=None, box=(0.105, 0.155, 0.855, 0.53), center=False, foot=True):
    fig = plt.figure(figsize=(TPL_W, TPL_H), dpi=TPL_DPI); fig.patch.set_facecolor(TPL_BG)
    x, ha = (0.5, "center") if center else (0.055, "left")
    fig.text(x, 0.935, title, ha=ha, va="top", fontsize=fit(title), fontweight="bold", color=TPL_TEXT)
    if sub: fig.text(x, 0.840, sub, ha=ha, va="top", fontsize=fit(sub, F_SUB, 0.94), color=TPL_SUB)
    if foot: fig.text(0.055, 0.045, FOOT, ha="left", va="bottom", fontsize=F_FOOT, color=TPL_TICK)
    ax = fig.add_axes(box); ax.set_facecolor(TPL_BG); _clean(ax)
    return fig, ax

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=TPL_DPI, facecolor=TPL_BG, bbox_inches=Bbox.from_bounds(0, 0, TPL_W, TPL_H))
    plt.close(fig)

def vgrad(ax, x, heights, width, top, bottom, zorder=3):
    cmap = LinearSegmentedColormap.from_list("vg", [bottom, top])
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    hs = np.asarray(heights, dtype=float)
    bars = ax.bar(x, hs, width=width, color=bottom, edgecolor="none", zorder=zorder)
    g = np.linspace(0, 1, 256).reshape(-1, 1)
    for b in bars:
        h = b.get_height()
        im = ax.imshow(g, aspect="auto", cmap=cmap, origin="lower", zorder=zorder + 0.1, extent=[b.get_x(), b.get_x() + b.get_width(), min(0.0, h), max(0.0, h)])
        im.set_clip_path(b)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    return bars

def tw(label, fs): return sum((1.00 if ord(c) > 127 else 0.55) for c in label) * fs / 864.0

def legend_row(fig, x, y, items, fs=F_LEG, mark="square"):
    xx = x
    for label, color in items:
        if mark == "square":
            fig.add_artist(Rectangle((xx, y - 0.008), 0.016, 0.016, transform=fig.transFigure, facecolor=color, edgecolor="none"))
        else:
            fig.add_artist(Circle((xx + 0.008, y), 0.009, transform=fig.transFigure, facecolor=color, edgecolor="none"))
        fig.text(xx + 0.024, y, label, ha="left", va="center", fontsize=fs, color=TPL_SUB)
        xx += 0.024 + tw(label, fs) + 0.028

def place_labels(ax, xs, ys, names, sizes=None, fs=17, box=None):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    b = box or ax.get_position(); wpt = b.width * TPL_W * 72; hpt = b.height * TPL_H * 72
    ux, uy = (x1 - x0) / wpt, (y1 - y0) / hpt; take = []
    r = np.sqrt(np.asarray(sizes if sizes is not None else [400] * len(xs)) / np.pi)
    occl = [(x - rr * ux, y - rr * uy, x + rr * ux, y + rr * uy) for x, y, rr in zip(xs, ys, r)]
    cands = [(0, 12, "center", "bottom"), (14, 0, "left", "center"), (0, -12, "center", "top"), (-14, 0, "right", "center"), (11, 10, "left", "bottom"), (11, -10, "left", "top"), (-11, 10, "right", "bottom"), (-11, -10, "right", "top")]
    for x, y, name in zip(xs, ys, names):
        wp = len(name) * fs * 0.98; hp = fs * 1.05; picked = None
        for dx, dy, ha, va in cands:
            cx, cy = x + dx * ux, y + dy * uy
            if ha == "left": bx0, bx1 = cx, cx + wp * ux
            elif ha == "right": bx0, bx1 = cx - wp * ux, cx
            else: bx0, bx1 = cx - wp * ux / 2, cx + wp * ux / 2
            if va == "bottom": by0, by1 = cy, cy + hp * uy
            elif va == "top": by0, by1 = cy - hp * uy, cy
            else: by0, by1 = cy - hp * uy / 2, cy + hp * uy / 2
            if not any(bx0 < o[2] and bx1 > o[0] and by0 < o[3] and by1 > o[1] for o in occl + take):
                picked = (dx, dy, ha, va, (bx0, by0, bx1, by1)); break
        if picked is None:
            dx, dy, ha, va = cands[-1]; picked = (dx, dy, ha, va, (0, 0, 0, 0))
        dx, dy, ha, va, bb = picked
        ax.annotate(name, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, fontsize=fs, color=TPL_SUB, zorder=6)
        take.append(bb)

df = pd.read_csv(os.path.join(CLEAN_DIR, "orders_clean.csv"), parse_dates=["Order Date", "Ship Date"])
SNAP = pd.Timestamp("2018-12-31")

def m_abc(df):
    g = df.groupby(["Category", "Sub-Category"])["Profit"].sum().reset_index().sort_values("Profit", ascending=False)
    total = g.loc[g["Profit"] > 0, "Profit"].sum()
    g["CumShare"] = g["Profit"].clip(lower=0).cumsum() / total
    g["Class"] = np.where(g["CumShare"] <= 0.8, "A", np.where(g["CumShare"] <= 0.95, "B", "C"))
    return g.reset_index(drop=True)

def m_bcg(df):
    g = df.groupby(["Category", "Sub-Category"]).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).reset_index()
    g["Margin"] = g["Profit"] / g["Sales"]; g["RelShare"] = g["Sales"] / g["Sales"].max()
    y = df.groupby(["Sub-Category", "Order Year"])["Sales"].sum().unstack()
    g["Growth"] = ((y[2018] - y[2017]) / y[2017]).reindex(g["Sub-Category"]).values
    y_line = df.loc[df["Order Year"] == 2018, "Sales"].sum() / df.loc[df["Order Year"] == 2017, "Sales"].sum() - 1
    g["Quadrant"] = np.select([(g["RelShare"] >= 0.35) & (g["Growth"] >= y_line), (g["RelShare"] >= 0.35) & (g["Growth"] < y_line), (g["RelShare"] < 0.35) & (g["Growth"] >= y_line)], ["明星", "金牛", "问题"], default="瘦狗")
    return g, y_line

def m_rfm(df):
    r = df.groupby("Customer ID").agg(Name=("Customer Name", "last"), Segment=("Segment", "last"), R=("Order Date", lambda s: (SNAP - s.max()).days), F=("Order ID", "nunique"), M=("Sales", "sum"), Profit=("Profit", "sum")).reset_index()
    rm, fm, mm = r["R"].median(), r["F"].median(), r["M"].median()
    rs = np.where(r["R"] <= rm, "高", "低"); fs = np.where(r["F"] >= fm, "高", "低"); ms = np.where(r["M"] >= mm, "高", "低")
    naming = {("高", "高", "高"): "重要价值客户", ("低", "高", "高"): "重要唤回客户", ("高", "低", "高"): "重要发展客户", ("低", "低", "高"): "重要挽留客户", ("高", "高", "低"): "一般价值客户", ("低", "高", "低"): "一般保持客户", ("高", "低", "低"): "一般发展客户", ("低", "低", "低"): "一般挽留客户"}
    r["RFM"] = [naming[(a, b, c)] for a, b, c in zip(rs, fs, ms)]
    s = r.groupby("RFM").agg(客户数=("Customer ID", "count"), 销售额=("M", "sum"), 利润=("Profit", "sum")).reset_index()
    s["销售占比"] = s["销售额"] / s["销售额"].sum()
    s = s.sort_values("销售额", ascending=False).reset_index(drop=True)
    return r, s, (rm, fm, mm)

def m_cohort(df, periods=13):
    d = df[["Customer ID", "Order Date"]].copy(); d["M"] = d["Order Date"].dt.to_period("M")
    first = d.groupby("Customer ID")["M"].min(); d["Cohort"] = d["Customer ID"].map(first)
    d["Idx"] = [(m - c).n for m, c in zip(d["M"], d["Cohort"])]
    size = d.groupby("Cohort")["Customer ID"].nunique()
    act = d.groupby(["Cohort", "Idx"])["Customer ID"].nunique().rename("n").reset_index()
    act = act[act["Idx"] < periods]; act["rate"] = act["n"] / act["Cohort"].map(size)
    piv = act.pivot_table(index="Cohort", columns="Idx", values="rate", aggfunc="first")
    return piv.reindex(sorted(piv.index)).reindex(columns=range(periods)), size

def m_kmeans(r):
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler
    X = r[["R", "F", "M"]].copy(); X["M"] = np.log1p(X["M"]); X["F"] = np.log1p(X["F"])
    Z = StandardScaler().fit_transform(X)
    km = KMeans(n_clusters=4, random_state=42, n_init=10).fit(Z)
    out = r.copy(); out["Cluster"] = km.labels_
    prof = out.groupby("Cluster").agg(N=("Customer ID", "count"), R=("R", "mean"), F=("F", "mean"), M=("M", "mean"), Profit=("Profit", "sum")).reset_index()
    rank = prof.sort_values("M")["Cluster"].tolist(); mp = {c: i for i, c in enumerate(rank)}
    out["Cluster"] = out["Cluster"].map(mp); prof["Cluster"] = prof["Cluster"].map(mp)
    return out, prof.sort_values("Cluster").reset_index(drop=True)

def m_forecast(df, steps=6):
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    s = df.groupby("Order YM")["Sales"].sum(); s.index = pd.PeriodIndex(s.index, freq="M").to_timestamp(); s = s.asfreq("MS")
    y = s.values.astype(float); h = 6; tr, te = y[:-h], y[-h:]
    def sarima(train, n):
        res = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        fc = res.get_forecast(steps=n); return fc.predicted_mean, fc.conf_int(alpha=0.05)
    def hw(train, n):
        res = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12, initialization_method="estimated").fit()
        p = np.asarray(res.forecast(n)); sd = np.std(res.resid)
        return p, np.column_stack([p - 1.96 * np.sqrt(1 + np.arange(n)) * sd, p + 1.96 * np.sqrt(1 + np.arange(n)) * sd])
    res = {}
    for name, fn in [("SARIMA", sarima), ("HW", hw)]:
        pred, _ = fn(tr, h); res[name] = dict(pred=pred, mape=np.mean(np.abs((te - pred) / te)) * 100, fn=fn)
    best = min(res, key=lambda k: res[k]["mape"]); mean, ci = res[best]["fn"](y, steps)
    fut = pd.date_range(s.index[-1] + pd.offsets.MonthBegin(1), periods=steps, freq="MS")
    return dict(hist=pd.Series(y, index=s.index), fut_idx=fut, fut=np.asarray(mean), lo=np.asarray(ci)[:, 0], hi=np.asarray(ci)[:, 1], best=best, metric=res)

def m_rules(df, min_sup=0.008, min_conf=0.15, max_len=3):
    baskets = df.groupby("Order ID")["Sub-Category"].apply(frozenset).tolist(); n = len(baskets); cnt = {}
    for b in baskets:
        for it in b: cnt[frozenset([it])] = cnt.get(frozenset([it]), 0) + 1
    freq = {k: v for k, v in cnt.items() if v / n >= min_sup}; allf = dict(freq); k = 2
    while k <= max_len and freq:
        prev = list(freq); cands = {a | b for a, b in itertools.combinations(prev, 2) if len(a | b) == k}; freq = {}
        for c in cands:
            if any(frozenset(x) not in allf for x in itertools.combinations(c, k - 1)): continue
            m = sum(1 for b in baskets if c <= b)
            if m / n >= min_sup: freq[c] = m
        allf.update(freq); k += 1
    rows = []
    for items, m in allf.items():
        if len(items) < 2: continue
        for r in range(1, len(items)):
            for ante in itertools.combinations(items, r):
                a = frozenset(ante); c = items - a; sup_a = allf.get(a, 0) / n; conf = (m / n) / sup_a if sup_a else 0; sup_c = sum(1 for b in baskets if c <= b) / n
                if conf >= min_conf and sup_c: rows.append({"前项": " + ".join(sorted(a)), "后项": " + ".join(sorted(c)), "支持度": m / n, "置信度": conf, "提升度": conf / sup_c})
    return pd.DataFrame(rows).sort_values("提升度", ascending=False).reset_index(drop=True)

def m_discount(df):
    o = df.groupby("Order ID").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"), Discount=("Discount", "max")).reset_index()
    o["Margin"] = o["Profit"] / o["Sales"]
    from sklearn.linear_model import LinearRegression
    X = np.column_stack([o["Discount"], o["Discount"] ** 2]); lr = LinearRegression().fit(X, o["Margin"])
    b0 = lr.intercept_; b1, b2 = lr.coef_; r2 = lr.score(X, o["Margin"])
    roots = np.roots([b2, b1, b0]); pos = [x.real for x in roots if abs(x.imag) < 1e-6 and 0 < x.real < 1]
    return o, (b0, b1, b2, r2, min(pos) if pos else np.nan)

def m_funnel(df):
    o = df.groupby("Order ID").agg(md=("Discount", "max"), Sales=("Sales", "sum"), Profit=("Profit", "sum")); rows = []
    for lab, th in [("全部订单", -1), ("有折扣", 0), ("折扣>10%", 0.1), ("折扣>20%", 0.2), ("折扣>30%", 0.3)]:
        sub = o if th < 0 else o[o["md"] > th]; rows.append({"层级": lab, "订单数": len(sub), "占比": len(sub) / len(o), "利润率": sub["Profit"].sum() / sub["Sales"].sum()})
    return pd.DataFrame(rows)

def fig16_pareto(g):
    d = g.sort_values("Profit", ascending=False).reset_index(drop=True); total = d.loc[d["Profit"] > 0, "Profit"].sum(); cum = d["Profit"].clip(lower=0).cumsum() / total
    nA = int((d["Class"] == "A").sum())
    fig, ax = canvas("子类别利润帕累托图", f"{nA} 个子类贡献 80% 利润，{int((d['Profit'] < 0).sum())} 个子类利润为负", box=(0.075, 0.205, 0.835, 0.45))
    x = np.arange(len(d)); ax.set_xlim(-0.7, len(d) - 0.3); ymax = d["Profit"].max(); ax.set_ylim(-ymax * 0.30, ymax * 1.18)
    yt = np.arange(0, ymax * 1.05, 10000); ax.set_yticks(yt); ax.set_yticklabels([f"{v/1e4:.0f}" for v in yt]); fig.text(0.055, 0.665, "万美元", fontsize=F_LAB - 3, color=TPL_TICK); _clean(ax, grid_y=True)
    pos = d["Profit"] > 0; vgrad(ax, x[pos.values], d.loc[pos, "Profit"].values, 0.62, TPL_SKY, TPL_BLUE_D)
    ax.bar(x[~pos.values], d.loc[~pos, "Profit"].values, width=0.62, color=TPL_RED, edgecolor="none", zorder=3); ax.axhline(0, color=TPL_GREY, lw=1.2, zorder=2)
    ax.set_xticks(x); ax.set_xticklabels([SUB_SHORT.get(s, s) for s in d["Sub-Category"]], fontsize=16)
    ax.axvline(nA - 0.5, color=TPL_GOLD, lw=1.6, ls=(0, (5, 5)), zorder=5); ax.text(nA - 0.30, ymax * 1.10, f"A 类 {nA} 个", fontsize=F_LAB, color=TPL_GOLD, va="center", ha="left")
    ax2 = ax.twinx(); ax2.plot(x, cum * 100, color=TPL_GOLD, lw=2.8, marker="o", ms=6, mec=TPL_BG, mew=1.6, zorder=6)
    ax2.set_ylim(0, 105); ax2.set_yticks([0, 20, 40, 60, 80, 100]); ax2.tick_params(length=0, labelsize=F_TICK, colors=TPL_GOLD, pad=10)
    for sp in ax2.spines.values(): sp.set_visible(False)
    ax2.axhline(80, color=TPL_RED, lw=1.6, ls=(0, (6, 6)), zorder=4); ax2.text(len(d) - 0.6, 83, "80% 基准线", ha="right", fontsize=F_LAB, color=TPL_RED, va="bottom")
    legend_row(fig, 0.055, 0.118, [("累计利润占比（右轴，%）", TPL_GOLD)], mark="circle")
    save(fig, "fig16_子类别利润帕累托图")

def fig17_bcg(g, y_line):
    fig, ax = canvas("子类别波士顿矩阵", "横轴=相对市场份额，纵轴=2018 年增长率，气泡=销售额，色深=利润率", box=(0.095, 0.185, 0.685, 0.48))
    ax.axvspan(0.35, 1.10, color="#212650", zorder=0); ax.set_xlim(-0.05, 1.10); ax.set_ylim(g["Growth"].min() * 1.15 - 0.08, g["Growth"].max() * 1.18 + 0.05)
    cmap = LinearSegmentedColormap.from_list("mg", ["#B03A4A", TPL_GOLD, TPL_TEAL, TPL_BLUE])
    sc = ax.scatter(g["RelShare"], g["Growth"], s=120 + g["Sales"] / 700, c=g["Margin"] * 100, cmap=cmap, vmin=-15, vmax=45, edgecolor=TPL_TEXT, linewidth=1.6, alpha=0.95, zorder=4)
    ax.axhline(y_line, color=TPL_GREY, lw=1.6, ls=(0, (6, 6)), zorder=2); ax.axvline(0.35, color=TPL_GREY, lw=1.6, ls=(0, (6, 6)), zorder=2)
    for k, (px, py, ha, va) in {"问题": (0.014, 0.975, "left", "top"), "明星": (0.986, 0.975, "right", "top"), "瘦狗": (0.014, 0.025, "left", "bottom"), "金牛": (0.986, 0.025, "right", "bottom")}.items():
        ax.text(px, py, k, transform=ax.transAxes, fontsize=36, color="#2F3670", fontweight="bold", ha=ha, va=va, zorder=1)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0]); ax.set_yticks([-0.2, 0, 0.2, 0.4, 0.6]); ax.set_yticklabels([f"{v*100:.0f}%" for v in ax.get_yticks()])
    place_labels(ax, g["RelShare"].values, g["Growth"].values, [SUB_CN.get(s, s) for s in g["Sub-Category"]], sizes=120 + g["Sales"].values / 700, fs=17)
    cax = fig.add_axes([0.845, 0.245, 0.016, 0.36]); cb = fig.colorbar(sc, cax=cax)
    cb.outline.set_visible(False); cb.ax.tick_params(length=0, labelsize=F_TICK, colors=TPL_TICK); cb.set_label("利润率（%）", color=TPL_SUB, fontsize=F_LEG, labelpad=14)
    save(fig, "fig17_子类别波士顿矩阵")

def fig18_rfm(s):
    fig, ax = canvas("RFM 八分类客户结构", "柱=客户数，折线=销售额占比（按客户价值从高到低排列）", box=(0.065, 0.235, 0.86, 0.42))

    order = ["重要价值客户", "重要唤回客户", "重要发展客户", "重要挽留客户", 
             "一般价值客户", "一般保持客户", "一般发展客户", "一般挽留客户"]
    s = s.set_index("RFM").reindex(order).reset_index()

    labels = ["重要价值", "重要唤回", "重要发展", "重要挽留", 
              "一般价值", "一般保持", "一般发展", "一般挽留"]
    
    x = np.arange(len(s))
    ax.set_xlim(-0.65, len(s) - 0.35)
    ax.set_ylim(0, s["客户数"].max() * 1.62)
    _clean(ax, grid_y=True)

    bars = ax.bar(x, s["客户数"], width=0.56, 
                  color=[RFM_COLORS.get(k, TPL_BLUE) for k in s["RFM"]], 
                  edgecolor="none", zorder=3)

    for b, v in zip(bars, s["客户数"]): 
        ax.text(b.get_x() + b.get_width() / 2, v + s["客户数"].max() * 0.035, f"{v}", 
                ha="center", va="bottom", fontsize=F_LAB, color=TPL_TEXT, fontweight="bold", zorder=7)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=F_TICK - 1, rotation=15)

    ax2 = ax.twinx()
    ax2.plot(x, s["销售占比"] * 100, color=TPL_GOLD, lw=2.6, marker="o", ms=9, mec=TPL_BG, mew=2, zorder=6)

    for xi, v in zip(x, s["销售占比"] * 100):
        ax2.text(xi, v + 3, f"{v:.1f}%", ha="center", va="bottom", fontsize=13, color=TPL_GOLD)
        
    ax2.set_ylim(0, 100)
    ax2.set_yticks([0, 25, 50, 75, 100])
    ax2.set_yticklabels(["0%", "25%", "50%", "75%", "100%"])
    ax2.tick_params(length=0, labelsize=F_TICK, colors=TPL_GOLD, pad=10)
    for sp in ax2.spines.values(): sp.set_visible(False)
    
    save(fig, "fig18_RFM八分类客户结构")

def fig19_rfm_scatter(r, med):
    fig, ax = canvas("RFM 客户分布气泡图", "横轴越左越活跃，气泡=累计消费额，颜色=八分类", box=(0.075, 0.245, 0.87, 0.43))
    for k in ["重要价值客户", "重要唤回客户", "重要发展客户", "重要挽留客户", "一般价值客户", "一般保持客户", "一般发展客户", "一般挽留客户"]:
        gg = r[r["RFM"] == k]; ax.scatter(gg["R"], gg["F"], s=np.sqrt(gg["M"]) * 1.5, color=RFM_COLORS[k], alpha=0.62, edgecolor="none", zorder=3)
    ax.axvline(med[0], color=TPL_GREY, lw=1.4, ls=(0, (6, 6)), zorder=1); ax.axhline(med[1], color=TPL_GREY, lw=1.4, ls=(0, (6, 6)), zorder=1)
    ax.set_xlim(0, 760); ax.set_ylim(0, 20); _clean(ax, grid_y=True)
    ax.text(med[0] + 14, 18.4, f"R 中位数 {med[0]:.0f} 天", fontsize=F_LAB, color=TPL_TICK); ax.text(742, med[1] + 0.35, f"F 中位数 {med[1]:.0f} 单", fontsize=F_LAB, color=TPL_TICK, ha="right")
    legend_row(fig, 0.075, 0.150, [(RFM_SHORT[k], RFM_COLORS[k]) for k in ["重要价值客户", "重要唤回客户", "重要发展客户", "重要挽留客户"]], fs=16, mark="circle")
    legend_row(fig, 0.075, 0.098, [(RFM_SHORT[k], RFM_COLORS[k]) for k in ["一般价值客户", "一般保持客户", "一般发展客户", "一般挽留客户"]], fs=16, mark="circle")
    save(fig, "fig19_RFM客户分布气泡图")

def fig20_cohort(piv, size, rows=16):
    p = piv.head(rows); m = np.nan_to_num(p.values.astype(float) * 100, nan=0.0)
    fig, ax = canvas("同期群月度留存热力图", f"以首购月分组，M0 为首购当月，共 {piv.shape[1]} 期", box=(0.165, 0.175, 0.75, 0.50))
    cmap = LinearSegmentedColormap.from_list("co", [TPL_BG, "#0E3A6B", TPL_BLUE, TPL_SKY, TPL_GOLD])
    im = ax.imshow(m, cmap=cmap, aspect="auto", vmin=0, vmax=60)
    ax.set_xticks(range(p.shape[1])); ax.set_xticklabels([f"M{i}" for i in range(p.shape[1])], fontsize=15)
    ax.set_yticks(range(len(p))); ax.set_yticklabels([f"{str(c)[:7]}  n={int(size.get(c, 0))}" for c in p.index], fontsize=14)
    ax.tick_params(colors=TPL_TICK, length=0)
    for sp in ax.spines.values(): sp.set_visible(False)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            if m[i, j] > 0: ax.text(j, i, f"{m[i,j]:.0f}", ha="center", va="center", fontsize=12, color=TPL_BG if m[i, j] > 34 else TPL_SUB)

    cax = fig.add_axes([0.655, 0.72, 0.285, 0.018])
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.outline.set_visible(False)
    cb.set_ticks([0, 20, 40, 60])
    cb.ax.tick_params(length=0, labelsize=F_TICK - 3, colors=TPL_TICK, pad=3)
    cb.ax.set_title("留存率 %", color=TPL_TICK, fontsize=F_LAB - 4, pad=4, loc="left")
    
    save(fig, "fig20_同期群月度留存热力图")

def fig21_kmeans(prof, out):
    base = {"R": 1 / out["R"].mean(), "F": out["F"].mean(), "M": out["M"].mean()}; prof = prof.copy()
    prof["R_idx"] = (1 / prof["R"]) / base["R"]; prof["F_idx"] = prof["F"] / base["F"]; prof["M_idx"] = prof["M"] / base["M"]
    prof["Share"] = prof["M"] * prof["N"] / (prof["M"] * prof["N"]).sum(); lab = ["低价值", "成长型", "潜力型", "高价值"]; cols = [TPL_GREY, TPL_GOLD, TPL_TEAL, TPL_BLUE]
    fig, ax = canvas("K-Means 四簇客户画像", "指标为簇均值 ÷ 全体均值；虚线 1.0 即全体平均水平", box=(0.075, 0.235, 0.88, 0.42))
    met = [("R_idx", "近度指数"), ("F_idx", "频次指数"), ("M_idx", "金额指数")]; w = 0.19; x = np.arange(len(met))
    ax.set_xlim(-0.55, len(met) - 0.45); ax.set_ylim(0, max(prof[["R_idx", "F_idx", "M_idx"]].values.max() * 1.30, 1.6)); _clean(ax, grid_y=True)
    for i, row in prof.iterrows():
        vals = [row[k] for k, _ in met]; ax.bar(x + (i - 1.5) * w, vals, width=w, color=cols[i], edgecolor="none", label=f"簇{i} {lab[i]}", zorder=3)
        for xx, v in zip(x + (i - 1.5) * w, vals): ax.text(xx, v + 0.04, f"{v:.2f}", ha="center", va="bottom", fontsize=13, color=TPL_SUB)
    ax.axhline(1.0, color=TPL_RED, lw=1.6, ls=(0, (6, 6)), zorder=4); ax.set_xticks(x); ax.set_xticklabels([n for _, n in met], fontsize=F_TICK)
    legend_row(fig, 0.075, 0.152, [(f"簇{i} {lab[i]}（{int(prof.loc[i,'N'])} 人）", cols[i]) for i in prof.index[:2]], fs=16)
    legend_row(fig, 0.075, 0.100, [(f"簇{i} {lab[i]}（{int(prof.loc[i,'N'])} 人）", cols[i]) for i in prof.index[2:]], fs=16)
    save(fig, "fig21_KMeans四簇客户画像")

def fig22_forecast(fc):
    fig, ax = canvas("月度销售额预测", f"主模型 {fc['best']}，外推 6 期并给出 95% 置信区间", box=(0.075, 0.205, 0.87, 0.455))
    h = fc["hist"]; n = len(h)
    ax.plot(h.index, h.values / 1e4, color=TPL_SKY, lw=2.8, zorder=4)
    ax.fill_between(fc["fut_idx"], fc["lo"] / 1e4, fc["hi"] / 1e4, color=TPL_GOLD, alpha=0.20, zorder=2)
    ax.plot(fc["fut_idx"], fc["fut"] / 1e4, color=TPL_GOLD, lw=3.0, marker="o", ms=8, mec=TPL_BG, mew=1.8, zorder=5)
    ax.plot([h.index[-1], fc["fut_idx"][0]], [h.values[-1] / 1e4, fc["fut"][0] / 1e4], color=TPL_GOLD, lw=2.2, ls=(0, (4, 4)), zorder=4)
    ax.axvline(h.index[-1], color=TPL_GREY, lw=1.4, ls=(0, (6, 6)), zorder=1)
    ax.set_xlim(h.index[0], fc["fut_idx"][-1]); ymax = max(fc["hi"].max(), h.values.max()) / 1e4; ax.set_ylim(0, ymax * 1.22); _clean(ax, grid_y=True)
    ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.text(0.075, 0.672, "万美元", fontsize=F_LAB - 3, color=TPL_TICK)
    ax.text(h.index[-1], ymax * 1.13, " 预测起点", fontsize=F_LAB, color=TPL_TICK)
    m = fc["metric"]; ax.text(h.index[2], ymax * 1.13, f"{fc['best']} 回测 MAPE {m[fc['best']]['mape']:.1f}%", fontsize=F_LAB, color=TPL_GOLD)
    legend_row(fig, 0.075, 0.108, [("实际销售额", TPL_SKY), ("预测值", TPL_GOLD), ("95% 置信区间", "#8A7433")], fs=16)
    save(fig, "fig22_月度销售额预测")

def _cn(items): return " + ".join(SUB_CN.get(x, x) for x in str(items).split(" + "))

def fig23_rules(r):
    top = r.nlargest(10, "提升度").iloc[::-1].reset_index(drop=True)
    fig, ax = canvas("关联规则 TOP10 提升度", f"共 {len(r)} 条强规则（支持度≥0.8%，置信度≥15%），红线 = 1.0 无提升", box=(0.315, 0.185, 0.625, 0.48))
    y = np.arange(len(top)); cols = [TPL_GOLD if v == top["提升度"].max() else TPL_BLUE for v in top["提升度"]]
    ax.barh(y, top["提升度"], height=0.55, color=cols, edgecolor="none", zorder=3)
    vmax = top["提升度"].max(); xr = vmax * 1.58
    for yy, v, c in zip(y, top["提升度"], top["置信度"]):
        ax.text(v + 0.03, yy, f"{v:.2f}", va="center", ha="left", fontsize=F_LAB, color=TPL_TEXT, fontweight="bold", zorder=4)
        ax.text(xr, yy, f"置信度 {c*100:.0f}%", va="center", ha="right", fontsize=F_LAB - 6, color=TPL_TICK, zorder=4)
    ax.set_yticks(y); ax.set_yticklabels([f"{_cn(a)} → {_cn(b)}" for a, b in zip(top["前项"], top["后项"])], fontsize=16)
    ax.set_xlim(0, vmax * 1.68); ax.set_ylim(-0.7, len(top) - 0.3); _clean(ax)
    ax.xaxis.grid(True, color=TPL_GRID, lw=1.0, ls=(0, (7, 7))); ax.set_axisbelow(True); ax.axvline(1.0, color=TPL_RED, lw=1.6, ls=(0, (6, 6)), zorder=4)
    save(fig, "fig23_关联规则提升度")

def fig24_discount(o, coef):
    b0, b1, b2, r2, be = coef
    fig, ax = canvas("折扣—利润率弹性曲线", f"订单级二次回归 R²={r2:.2f}，拟合盈亏平衡点 ≈ {be*100:.0f}%", box=(0.08, 0.215, 0.87, 0.445))
    ax.scatter(o["Discount"], o["Margin"] * 100, s=14, color=TPL_SKY, alpha=0.16, edgecolor="none", zorder=2)
    xs = np.linspace(0, 0.8, 240); ax.plot(xs, (b0 + b1 * xs + b2 * xs ** 2) * 100, color=TPL_GOLD, lw=3.4, zorder=6)
    ob = o.assign(band=pd.cut(o["Discount"], np.arange(0, 0.85, 0.05)))
    gb = ob.groupby("band", observed=True).agg(m=("Margin", "mean"), n=("Margin", "size")); gb = gb[gb["n"] >= 10]
    ax.plot([iv.mid for iv in gb.index], gb["m"].values * 100, color=TPL_CYAN, marker="o", ms=9, lw=2.2, mec=TPL_BG, mew=1.8, zorder=7)
    ax.axvline(0.2, color=TPL_RED, lw=1.6, ls=(0, (6, 6)), zorder=5); ax.axvline(be, color=TPL_CYAN, lw=1.8, ls=(0, (2, 3)), zorder=5); ax.axhline(0, color=TPL_GREY, lw=1.4, zorder=3)
    ax.set_xlim(-0.02, 0.82); ax.set_ylim(-115, 62); ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8]); ax.set_xticklabels(["0%", "20%", "40%", "60%", "80%"]); _clean(ax, grid_y=True)
    ax.text(0.205, 44, "实证预警线 20%", fontsize=F_LAB, color=TPL_RED); ax.text(be + 0.016, 30, f"盈亏平衡 {be*100:.0f}%", fontsize=F_LAB, color=TPL_CYAN)
    legend_row(fig, 0.08, 0.115, [("订单样本", TPL_SKY), ("二次拟合", TPL_GOLD), ("5% 分箱均值", TPL_CYAN)], fs=16, mark="circle")
    save(fig, "fig24_折扣利润率弹性曲线")

def fig25_funnel(f):
    fig, ax = canvas("折扣深度漏斗（订单层级）", "折扣越深订单越少，利润率同步由正转负", box=(0.055, 0.135, 0.89, 0.545))
    ax.set_xlim(0, 1.0); ax.set_ylim(-0.55, len(f) - 0.45); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    cols = [TPL_BLUE_D, TPL_BLUE, TPL_SKY, TPL_GOLD, TPL_RED]; 

    x0, wmax = 0.14, 0.40
    
    ax.text(x0 + wmax + 0.02, len(f) - 0.52, "订单数与占比", va="center", fontsize=F_LAB - 4, color=TPL_TICK); ax.text(0.995, len(f) - 0.52, "利润率", ha="right", va="center", fontsize=F_LAB - 4, color=TPL_TICK)
    for i, r in f.iterrows():
        y = len(f) - 1 - i; w = r["占比"] / f["占比"].iloc[0] * wmax
        ax.barh(y, w, left=x0, height=0.55, color=cols[i], edgecolor="none", zorder=3)
        ax.text(x0 - 0.03, y, r["层级"], ha="right", va="center", fontsize=20, color=TPL_TEXT, zorder=4)

        ax.text(x0 + wmax + 0.02, y, f"{r['订单数']:,} 单 ｜ {r['占比']*100:.0f}%", va="center", ha="left", fontsize=F_LAB, color=TPL_TEXT, zorder=4)
        ax.text(0.995, y, f"{r['利润率']*100:+.1f}%", va="center", ha="right", fontsize=22, color=TPL_GOLD if r["利润率"] < 0 else TPL_CYAN, fontweight="bold", zorder=4)
    
    save(fig, "fig25_折扣深度漏斗")

g = m_abc(df); fig16_pareto(g)
bc, y_line = m_bcg(df); fig17_bcg(bc, y_line)
rfm, summ, med = m_rfm(df); fig18_rfm(summ); fig19_rfm_scatter(rfm, med)
piv, size = m_cohort(df); fig20_cohort(piv, size)
out, prof = m_kmeans(rfm); fig21_kmeans(prof, out)
fc = m_forecast(df); fig22_forecast(fc)
rules = m_rules(df); fig23_rules(rules)
o, coef = m_discount(df); fig24_discount(o, coef)
fun = m_funnel(df); fig25_funnel(fun)

for name, t in [("t32_abc", g), ("t34_bcg", bc), ("t36_rfm", summ), ("t37_cohort", piv.reset_index()), ("t38_kmeans", prof), ("t42_rules", rules), ("t45_funnel", fun)]:
    t.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=False, encoding="utf-8-sig")
print("业务模型图完成")

业务模型图完成
